# DIMER Notebook: OCR and Structured Document Extraction
## GOT-OCR 2.0 vs SmolDocling

**Profile:** `TASK-INFERENCE` · **Mode:** `WORKSHOP` · **Notebook Spec:** `2.1` · **Standalone:** yes

This notebook compares **text-first OCR** with **structured document conversion**:

- **GOT-OCR 2.0** → plain or formatted generated text.
- **SmolDocling** → DocTags carrying text, typed document elements, location tokens, reading order and OTSL tables.

The common quantitative path measures text recovery. Native structure is analyzed separately rather than forcing incompatible formats into one schema.

**AI Use Disclosure:** Generative AI assisted with this notebook’s code and instructional content under maintainer direction. The maintainer remains responsible for review, validation, and release decisions. AI-generated material may contain errors; validation claims are limited to documented runs and configurations. AI use does not imply independent verification, provider endorsement, or release approval.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and are new to OCR or multimodal document extraction.

**Runtime.** Use the documented GPU runtime; the default path loads the large OCR/document models sequentially.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; leave the default settings unchanged on your first pass.
3. Read the explanatory markdown while the notebook runs.
4. Sections marked **Infrastructure** support reproducibility, model acquisition, or orchestration. Run those cells as written; understanding their implementation is not a learning objective.

### Task at a glance

`document image → OCR / document model → text or structured representation → fidelity/structure evaluation`

### Roadmap

1. Understand the task and its input/output contract.
2. Inspect and validate the built-in data or inputs.
3. Establish the baseline/reference behavior.
4. Run the model or multi-model comparison.
5. Inspect errors, disagreements, robustness, and/or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion; optionally continue with BYOD.

### What successful execution looks like

You should finish with a validated input/sample, the notebook's principal baseline/reference, model outputs and evaluation results, at least one diagnostic or qualitative comparison, and machine-readable results/provenance where supported. Exact values can vary slightly across supported runtimes; focus on the defined metrics and the observed pattern.

**Reading layers:** core concepts cover image-to-text, DocTags and structures; evaluation practice covers baselines, metrics and controlled probes; infrastructure downloads and runtime classes are collapsed and can remain collapsed. Basic Python and Colab familiarity is sufficient. Run section by section for learning and predict before revealing results. The budget activity reads already-computed probes without reloading a released model or changing the canonical result.


## 1. OCR versus document extraction

OCR asks **what text is visible**. Structured extraction also asks **what kind of element contains it, where it is, and how page elements relate**.

A transcript can be textually accurate but structurally weak; a structured conversion can preserve layout while making transcription errors.

## 2. Model contracts

### GOT-OCR 2.0
`stepfun-ai/GOT-OCR-2.0-hf` — Apache-2.0, 560.5M parameters. A SAM-style ViT feeds a Qwen2-style decoder. Inputs are resized to **1024×1024 without preserving aspect ratio**. DIMER exposes `plain` and `format`.

### SmolDocling
`docling-project/SmolDocling-256M-preview` — CDLA-Permissive-2.0, 256.5M parameters. A SigLIP-style visual encoder feeds a SmolLM2 decoder. Pages are processed with **longest edge ~2048 px, 512-px tiles plus a global view**. The native output is DocTags.

## 3. Configuration

In [ ]:
USE_BYOD=False  # @param {type:"boolean"}
BYOD_PATH=""  # @param {type:"string"}

RUN_BELFORT=True  # @param {type:"boolean"}
RUN_PAGE_SUITE=True  # @param {type:"boolean"}
RUN_TOKEN_BUDGET_EXPERIMENT=True  # @param {type:"boolean"}
RUN_SHAPE_ROBUSTNESS=True  # @param {type:"boolean"}

GOT_LINE_MAX_NEW_TOKENS=128
GOT_PAGE_MAX_NEW_TOKENS=1024
SMOLDOC_LINE_MAX_NEW_TOKENS=160
SMOLDOC_PAGE_MAX_NEW_TOKENS=2048
BATCH_SIZE=8

MIN_IMAGE_SIDE=16
MAX_IMAGE_SIDE=16384
MAX_IMAGE_PIXELS=4096*4096
OUTPUT_DIR="outputs/document_extraction"

## 4. Runtime

The current carrier runtime is reproduced: Python 3.12, PyTorch 2.14, Transformers 4.57.6, SafeTensors, Pillow, Hugging Face Hub and PyArrow. Both models run in float32.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

Runtime pins are compared by their PEP 440 public version, allowing CUDA local suffixes. If installed packages differ from pre-imported modules, use **Runtime → Restart session**, retaining installed packages, then Run all. Such execution is restart-assisted; uninterrupted fresh-runtime qualification remains pending.


In [ ]:
import importlib.metadata as importlib_metadata, subprocess, sys
from packaging.version import Version, InvalidVersion
PINS={
 "torch":"2.14.0","torchvision":"0.29.0","torchaudio":"2.11.0",
 "transformers":"4.57.6","safetensors":"0.8.0","numpy":"2.1.3",
 "pillow":"11.3.0","huggingface-hub":"0.36.2","pyarrow":"25.0.1"
}
def dv(n):
    try:return importlib_metadata.version(n)
    except importlib_metadata.PackageNotFoundError:return None
def matches_public_version(observed, expected):
    if observed is None: return False
    try: return Version(Version(str(observed)).public) == Version(Version(str(expected)).public)
    except InvalidVersion: return False

needed=[f"{k}=={v}" for k,v in PINS.items() if not matches_public_version(dv(k),v)]
if needed: subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*needed])
after={name:dv(name) for name in PINS}
bad={name:(after[name],pin) for name,pin in PINS.items() if not matches_public_version(after[name],pin)}
if bad: raise RuntimeError(f"Pinned installation did not converge: {bad}")
stale=[]
for module_name,dist_name in (("torch","torch"),("numpy","numpy"),("transformers","transformers"),("pyarrow","pyarrow")):
    module=sys.modules.get(module_name)
    observed=getattr(module,"__version__",None) if module is not None else None
    if observed is not None and not matches_public_version(observed,after[dist_name]):
        stale.append((module_name,str(observed),after[dist_name]))
if stale:
    raise RuntimeError("Installed packages differ from loaded modules. Use Runtime > Restart session "
                       "to retain installed packages, then Run all. This is restart-assisted, not "
                       f"uninterrupted fresh-runtime qualification. Stale modules: {stale}")

import csv, gc, hashlib, io, json, random, re, time, urllib.request, zipfile
from collections import Counter
from pathlib import Path
import numpy as np, torch, pyarrow.parquet as pq
from PIL import Image, ImageDraw, ImageFont
from transformers import AutoModelForImageTextToText, AutoProcessor
from huggingface_hub import hf_hub_download

DEVICE="cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float32
RUNTIME={"python":sys.version.split()[0],"torch":torch.__version__,
 "transformers":importlib_metadata.version("transformers"),
 "numpy":np.__version__,"pyarrow":importlib_metadata.version("pyarrow"),
 "device":DEVICE,"dtype":"float32"}
if torch.cuda.is_available(): RUNTIME["gpu_name"]=torch.cuda.get_device_name(0)
print(RUNTIME)

## 5. Immutable model snapshots

Only manifest-listed files at immutable revisions are fetched. Every file is verified by byte size and SHA-256 before local loading with `trust_remote_code=False`.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
GOT_MANIFEST=json.loads(r'''{"format":"dimer_hf_snapshot","formatVersion":1,"modelKey":"got-ocr-2.0-hf","modelId":"stepfun-ai/GOT-OCR-2.0-hf","revision":"d3017ef2c2c1395888c8d635c5e0508bcb0ac78d","files":[{"path":"README.md","bytes":12077,"sha256":"706d6fe217d2f047ca68f47bdcf44ace4a0832491da396972c2c917944e84c18"},{"path":"config.json","bytes":608,"sha256":"cbe8aacd6cd84a2d58eafcd0045c6ac40e02e3a448f24b8cee51cc81d8bdccf2"},{"path":"generation_config.json","bytes":74,"sha256":"31915c5a692f43c5765a20cfc5f9403bcd250f5721a0d931bb703169c08993b4"},{"path":"model.safetensors","bytes":1121114488,"sha256":"6175ac7868a4e75735f5d59f78c465081ad3427eb4f312d072a0f1d16b333ba4"},{"path":"preprocessor_config.json","bytes":439,"sha256":"ef9a0dc0935cac11f4230ca30d00a52bedfa52b6633e409e9fbd2ea56373aa7e"},{"path":"special_tokens_map.json","bytes":213,"sha256":"7c2368a3889fdfb37c24cabeb031b53f47934f357b54e56e8e389909a338ea47"},{"path":"tokenizer.json","bytes":18702549,"sha256":"36b382a3c48c9a143c30139dac6c8230ddfb0b46a3dc43082af6052abe99d9de"},{"path":"tokenizer_config.json","bytes":39228,"sha256":"8b0542937d32a67da8ea2d1288b870e325be383a962c65d201864299560a2b8e"}],"totalBytes":1139869676}''')
SMOL_MANIFEST=json.loads(r'''{"format":"dimer_hf_snapshot","formatVersion":1,"modelKey":"smoldocling-256m-preview","modelId":"docling-project/SmolDocling-256M-preview","revision":"ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8","files":[{"path":"README.md","bytes":16108,"sha256":"9b82c4dd1b38340656da55d628d63bf319c5fc4700811148687b6d8070e7e493"},{"path":"added_tokens.json","bytes":3667,"sha256":"fc79a032b551636ad0fe6c0e16bfe38c43b5843895cbb0544a4a5919818472cc"},{"path":"chat_template.json","bytes":430,"sha256":"b585e3598909a5687f9f9d738d35223724dedef256b9b274e1cbfb32b13c74bf"},{"path":"config.json","bytes":3903,"sha256":"57af2810c65b9896a8d1d65c67aabdb9296d497aa10a6329f4bb2ddce623586f"},{"path":"generation_config.json","bytes":141,"sha256":"0758109c85e7f7d6b0202ebf643bb07c5625b3363e389854b414d9a701becc28"},{"path":"merges.txt","bytes":466391,"sha256":"0b54e8aa4e53d5383e2e4bc635a56b43f9647f7b13832d5d9ecd8f82dac4f510"},{"path":"model.safetensors","bytes":513028808,"sha256":"cdcdf5d823c5684029c7d8e52177cf10f9034b3aba6577549cfb1a9ce36ad0a2"},{"path":"preprocessor_config.json","bytes":486,"sha256":"6cb6e36d6fcb88ca1502c4a26750715dc3e7dedddc9a8f17b27d8d167d1457e7"},{"path":"processor_config.json","bytes":68,"sha256":"e7bff42da73ae9eec9042ef20e066e11f1ee20f025358ff79131e3c0fb549b46"},{"path":"special_tokens_map.json","bytes":1069,"sha256":"aa0ff906077086dfa9734a7f97f68c825877a48f9468807be65504495cdeef09"},{"path":"tokenizer.json","bytes":3547443,"sha256":"7c5cf6233a3dc8b9e54fb729ee6e771bdf5f0d65fd9075b5e60bed837959deee"},{"path":"tokenizer_config.json","bytes":27362,"sha256":"b38f39506a4fa7d2604015a6c303df320675cfeb7ba1f5969e1c44032727107b"},{"path":"vocab.json","bytes":800662,"sha256":"82b84012e3add4d01d12ba14442026e49b8cbbaead1f79ecf3d919784f82dc79"}],"totalBytes":517896538}''')
SNAPSHOT_ROOT=Path("weights/document-extraction"); SNAPSHOT_ROOT.mkdir(parents=True,exist_ok=True)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()

def stage_snapshot(manifest):
    root=SNAPSHOT_ROOT/manifest["modelKey"]; root.mkdir(parents=True,exist_ok=True)
    (root/"dimer-base-manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    t=time.perf_counter()
    for e in manifest["files"]:
        p=root/e["path"]
        if not p.is_file():
            p.parent.mkdir(parents=True,exist_ok=True)
            hf_hub_download(repo_id=manifest["modelId"],filename=e["path"],
                            revision=manifest["revision"],local_dir=str(root))
    for e in manifest["files"]:
        p=root/e["path"]
        if p.stat().st_size!=e["bytes"] or sha256_file(p)!=e["sha256"]:
            raise RuntimeError(f"snapshot verification failed: {manifest['modelKey']}/{e['path']}")
    return root,time.perf_counter()-t

GOT_DIR,got_verify_seconds=stage_snapshot(GOT_MANIFEST)
SMOL_DIR,smol_verify_seconds=stage_snapshot(SMOL_MANIFEST)
print("Snapshots verified.")

## 6. Belfort-line provenance

The common real-world transcription sample is the exact digest-pinned Belfort sample shared by both live carriers: eight row groups from `Teklia/Belfort-line` at immutable parquet-conversion revision `c4a74bbd…`.

Only the required row groups are read over HTTPS range requests.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
ROW_GROUP_PINS=json.loads(r'''{"0":["1dc3141e4809ea628b17c3ca7b81d64e6ca92bce18dd5765ecd618bfc7867954",5481145],"1":["c9d3b52013933c803f4886edbce68da0ae483347a4a6ff3e0f5ced1db4a7e653",5465901],"2":["6e0578a90a07a9e25e65b765881d3fa33d6a797624425e01026980d7287f0bf6",5379166],"3":["00cdfb7aabe924f31b9f1bb1ba4849040051e5619567b68bf99fdcbcab15131a",5821303],"4":["fc063442fb20e7a60c2533ab44dcc69a22ad59f5ce24921fe6af5f53ceab7e1a",5163559],"5":["2d7e29331bd4e93e0c8a1caa9a83b8f1ede9b17af6dae9377b83f56c56f05689",4713140],"6":["49423d91780cb184c4b0069630e85acccb314a113256ced9692a5138c4782ef1",5069794],"7":["3a010831456f185399579b4ecf9d46222f95368c3cdbbc2ff103a258b9c16e2f",5309891]}''')
ROW_GROUP_PINS={int(k):(v[0],int(v[1])) for k,v in ROW_GROUP_PINS.items()}
CORPUS_REPO="Teklia/Belfort-line"
CORPUS_REVISION="c4a74bbd39f2df314752e7e6026649a39d365cbb"
CORPUS_FILE="default/test/0000.parquet"
CORPUS_BYTES=210_579_166
CORPUS_ROWS=3_819
CORPUS_URL=f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
BELFORT_CACHE=Path("weights/belfort"); BELFORT_CACHE.mkdir(parents=True,exist_ok=True)
SAMPLE_SEED=42
SAMPLE_SPLIT={"train":600,"validation":60,"test":140}
SAMPLE_DIGEST="b7e1dd684691a0eedb63a609311f4964e7732e5c1a8d254fe4e1293a8cd0964d"

def normalise_text(text): return " ".join(str(text).split())

class HttpRangeFile(io.RawIOBase):
    def __init__(self,url,size): self.url,self.size,self.pos=url,size,0
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self,offset,whence=0):
        self.pos=max(0,{0:0,1:self.pos,2:self.size}[whence]+offset); return self.pos
    def read(self,n=-1):
        if n is None or n<0: n=self.size-self.pos
        if n<=0 or self.pos>=self.size: return b""
        end=min(self.size,self.pos+n)-1
        req=urllib.request.Request(self.url,headers={"Range":f"bytes={self.pos}-{end}",
                                                     "User-Agent":"dimer-doc-extract/1.0"})
        with urllib.request.urlopen(req,timeout=300) as response:
            if response.status!=206: raise ValueError(f"server ignored Range request: {response.status}")
            data=response.read()
        self.pos+=len(data); return data
    def readinto(self,buffer):
        data=self.read(len(buffer)); buffer[:len(data)]=data; return len(data)

def declared_size(url):
    req=urllib.request.Request(url,method="HEAD",headers={"User-Agent":"dimer-doc-extract/1.0"})
    with urllib.request.urlopen(req,timeout=60) as response: value=response.headers.get("Content-Length")
    if value is None: raise ValueError("missing Content-Length")
    return int(value)

def group_digest(rows):
    h=hashlib.sha256(); total=0
    for row in rows:
        b=row["image"]["bytes"]; t=str(row["text"]).encode("utf-8")
        h.update(b); h.update(t); total+=len(b)+len(t)
    return h.hexdigest(),total

def fetch_groups():
    if declared_size(CORPUS_URL)!=CORPUS_BYTES: raise RuntimeError("Belfort shard size drift")
    reader=pq.ParquetFile(HttpRangeFile(CORPUS_URL,CORPUS_BYTES))
    if reader.metadata.num_rows!=CORPUS_ROWS: raise RuntimeError("Belfort row-count drift")
    out={}
    for group in sorted(ROW_GROUP_PINS):
        local=BELFORT_CACHE/f"test-rg{group}.parquet"; rows=None
        if local.is_file():
            candidate=pq.read_table(local).to_pylist()
            if group_digest(candidate)==ROW_GROUP_PINS[group]: rows=candidate
        if rows is None:
            table=reader.read_row_group(group,columns=["image","text"])
            rows=table.to_pylist()
            if group_digest(rows)!=ROW_GROUP_PINS[group]: raise RuntimeError(f"row group {group} digest drift")
            pq.write_table(table,local)
        out[group]=rows
    return out

def image_digest(image):
    rgb=image.convert("RGB")
    return hashlib.sha256(f"{rgb.width}x{rgb.height}:".encode()+rgb.tobytes()).hexdigest()

def read_corpus(groups):
    out=[]
    for group in sorted(groups):
        for idx,row in enumerate(groups[group]):
            text=normalise_text(row["text"])
            if not text: continue
            image=Image.open(io.BytesIO(row["image"]["bytes"])); image.load()
            out.append({"id":f"belfort-test-{group*100+idx}","image":image.convert("RGB"),
                         "text":text,"source_row_group":group})
    return out

def dataset_digest(records):
    parts=sorted(f"{r['id']}:{image_digest(r['image'])}:{normalise_text(r['text'])}" for r in records)
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()

def build_split(records):
    pool=[dict(r) for r in records]; random.Random(SAMPLE_SEED).shuffle(pool)
    out={}; cursor=0
    for name,count in SAMPLE_SPLIT.items():
        out[name]=pool[cursor:cursor+count]; cursor+=count
    return out

belfort_splits=build_split(read_corpus(fetch_groups()))
combined=[r for part in belfort_splits.values() for r in part]
if dataset_digest(combined)!=SAMPLE_DIGEST: raise RuntimeError("Belfort sample digest mismatch")
seen={}
for name,part in belfort_splits.items():
    for r in part:
        d=image_digest(r["image"])
        if d in seen and seen[d]!=name: raise RuntimeError("cross-split image leakage")
        seen[d]=name
test_records=belfort_splits["test"]
print({"split_sizes":{k:len(v) for k,v in belfort_splits.items()},"digest":dataset_digest(combined)})

## 7. Common OCR metrics and baselines

CER/WER are uncapped. A score above 1.0 is valid when generated insertions exceed the reference length.

The constant baseline predicts the medoid of the first 120 training transcripts.

> **Before you run it:** predict whether this simple reference will be easy or difficult for the learned model(s) to beat. Record the baseline before interpreting the more complex result.

In [ ]:
def edit_distance(reference,hypothesis):
    previous=list(range(len(hypothesis)+1))
    for i,a in enumerate(reference,1):
        current=[i]
        for j,b in enumerate(hypothesis,1):
            current.append(min(current[-1]+1,previous[j]+1,previous[j-1]+(a!=b)))
        previous=current
    return previous[-1]

def words(text): return normalise_text(text).lower().split()

def one_metrics(reference,hypothesis):
    ref=normalise_text(reference); hyp=normalise_text(hypothesis)
    ce=edit_distance(ref,hyp); rw,hw=words(ref),words(hyp); we=edit_distance(rw,hw)
    return {"cer":ce/len(ref),"wer":we/len(rw),"exact_match":hyp==ref,
            "reference_chars":len(ref),"hypothesis_chars":len(hyp),
            "length_ratio":len(hyp)/len(ref),"char_edits":ce,"word_edits":we,
            "reference_words":len(rw)}

def corpus_metrics(hypotheses,records):
    rows=[{"id":r["id"],**one_metrics(r["text"],h)}
          for h,r in zip(hypotheses,records,strict=True)]
    return {"n":len(rows),
            "cer":sum(x["char_edits"] for x in rows)/sum(x["reference_chars"] for x in rows),
            "wer":sum(x["word_edits"] for x in rows)/sum(x["reference_words"] for x in rows),
            "cer_macro":float(np.mean([x["cer"] for x in rows])),
            "wer_macro":float(np.mean([x["wer"] for x in rows])),
            "exact_match":float(np.mean([x["exact_match"] for x in rows])),
            "median_cer":float(np.median([x["cer"] for x in rows])),
            "p90_cer":float(np.quantile([x["cer"] for x in rows],0.9)),
            "ref_chars":sum(x["reference_chars"] for x in rows),
            "hyp_chars":sum(x["hypothesis_chars"] for x in rows),
            "length_ratio":sum(x["hypothesis_chars"] for x in rows)/sum(x["reference_chars"] for x in rows),
            "rows":rows}

def medoid_transcript(train,pool=120):
    texts=[normalise_text(r["text"]) for r in train[:pool] if normalise_text(r["text"])]
    return min(texts,key=lambda c:sum(edit_distance(other,c)/len(other) for other in texts))

empty_metrics=corpus_metrics([""]*len(test_records),test_records)
constant_text=medoid_transcript(belfort_splits["train"])
constant_metrics=corpus_metrics([constant_text]*len(test_records),test_records)
print({"empty_cer":empty_metrics["cer"],"constant_cer":constant_metrics["cer"],"constant":constant_text})

## 8. Deterministic rendered pages

The notebook renders four pages in code: Notice, Report with tables, Invoice-style document, and Technical Note. Each carries known visible text and authored SmolDocling structure expectations.

In [ ]:
PAGE_DIR=Path(OUTPUT_DIR)/"pages"; PAGE_DIR.mkdir(parents=True,exist_ok=True)
def fnt(size): return ImageFont.load_default(size=size)
def wrapped(text,n):
    out=[]; line=[]
    for w in text.split():
        if len(" ".join(line+[w]))>n and line: out.append(" ".join(line)); line=[w]
        else: line.append(w)
    if line: out.append(" ".join(line))
    return out
def block(draw,text,x,y,width,size=22):
    lines=wrapped(text,width)
    for line in lines:
        draw.text((x,y),line,fill="black",font=fnt(size)); y+=size+8
    return y
def table(draw,x,y,widths,rows,row_h=40,size=18):
    W=sum(widths); H=len(rows)*row_h
    draw.rectangle((x,y,x+W,y+H),outline="black",width=2)
    xx=x
    for w in widths[:-1]: xx+=w; draw.line((xx,y,xx,y+H),fill="black")
    for i in range(1,len(rows)): draw.line((x,y+i*row_h,x+W,y+i*row_h),fill="black")
    for i,row in enumerate(rows):
        xx=x
        for j,val in enumerate(row):
            draw.text((xx+5,y+i*row_h+8),str(val),fill="black",font=fnt(size)); xx+=widths[j]
    return y+H

def notice():
    im=Image.new("RGB",(1000,1300),"white"); d=ImageDraw.Draw(im)
    d.text((70,50),"PUBLIC NOTICE",fill="black",font=fnt(42))
    d.text((70,115),"Document Processing Workshop — 26 September 2026",fill="black",font=fnt(21))
    ps=["The records office will conduct a scheduled document digitization activity this Friday.",
        "Participants should bring one sample page and verify all extracted text against the original document.",
        "Generated OCR output may contain omissions, substitutions, repetitions, or invented text and must be reviewed."]
    y=200
    for p in ps: y=block(d,p,70,y,72,23)+20
    bullets=["Prepare the source document.","Check the recognized text.","Review the exported structure."]
    for b in bullets: d.text((95,y),f"• {b}",fill="black",font=fnt(23)); y+=42
    footer="DIMER Document Intelligence Workshop"; d.text((70,1200),footer,fill="black",font=fnt(20))
    return im,normalise_text(" ".join(["PUBLIC NOTICE","Document Processing Workshop — 26 September 2026",*ps,*bullets,footer])),{"section_header_level_1":1,"text":4,"page_footer":1}

def report():
    im=Image.new("RGB",(1100,1500),"white"); d=ImageDraw.Draw(im)
    d.text((70,45),"DOCUMENT EXTRACTION REPORT",fill="black",font=fnt(38))
    p1="This report summarizes a controlled document conversion exercise using synthetic text and deterministic tables."
    p2="The page is generated locally so every visible word and table cell has a known reference."
    y=120; y=block(d,p1,70,y,82,22)+18; y=block(d,p2,70,y,82,22)+30
    r1=[["Item","Q1","Q2","Q3","Total"]]+[[f"Series {i}",i,i+1,i+2,3*i+3] for i in range(1,8)]
    y=table(d,70,y,[210,130,130,130,150],r1,38)+40
    r2=[["Code","Count","Status"]]+[[f"A{i}",i*3,"Ready" if i%2 else "Review"] for i in range(1,5)]
    y=table(d,70,y,[240,170,300],r2,42)+35
    close="End of report. Verify extracted text and structural markup before downstream use."
    block(d,close,70,y,82,22)
    parts=["DOCUMENT EXTRACTION REPORT",p1,p2]
    for row in r1+r2: parts+=list(map(str,row))
    parts.append(close)
    return im,normalise_text(" ".join(parts)),{"section_header_level_1":1,"text":3,"otsl":2}

def invoice():
    im=Image.new("RGB",(1000,1350),"white"); d=ImageDraw.Draw(im)
    d.text((70,45),"INVOICE",fill="black",font=fnt(44))
    fields=[("Invoice Number","INV-2026-0926"),("Date","26 September 2026"),
            ("Customer","Sample Research Office"),("Address","C.P. Garcia Avenue, Diliman, Quezon City")]
    y=130
    for k,v in fields: d.text((70,y),f"{k}: {v}",fill="black",font=fnt(21)); y+=42
    rows=[["Description","Qty","Unit Price","Amount"],["Document scan",3,"120.00","360.00"],
          ["OCR review",2,"250.00","500.00"],["Structure check",1,"400.00","400.00"]]
    y=table(d,70,y+20,[360,100,170,170],rows,44)+35
    totals=[("Subtotal","1,260.00"),("Tax","151.20"),("Total","1,411.20")]
    for k,v in totals: d.text((560,y),f"{k}: {v}",fill="black",font=fnt(23)); y+=42
    note="Payment note: This synthetic invoice is for OCR and document extraction testing only."
    block(d,note,70,y+30,74,21)
    parts=["INVOICE"]
    for k,v in fields+totals: parts += [k,v]
    for row in rows: parts+=list(map(str,row))
    parts.append(note)
    return im,normalise_text(" ".join(parts)),{"section_header_level_1":1,"text":8,"otsl":1,"key_value_region":1}

def technical():
    im=Image.new("RGB",(1050,1300),"white"); d=ImageDraw.Draw(im)
    d.text((70,45),"TECHNICAL NOTE",fill="black",font=fnt(40))
    intro="The following expressions are rendered as plain text for deterministic extraction testing."
    y=block(d,intro,70,125,76,22)+35
    f1="Energy: E = m * c^2"; f2="Mean: x_bar = (x1 + x2 + x3) / 3"
    d.text((100,y),f1,fill="black",font=fnt(29)); y+=70
    d.text((100,y),f2,fill="black",font=fnt(29)); y+=85
    note="Measured value: 12.5 kJ. Tolerance: plus or minus 0.2 kJ."
    y=block(d,note,70,y,76,22)+35
    close="The formulas above are instructional content, not production measurements."
    block(d,close,70,y,76,22)
    return im,normalise_text(" ".join(["TECHNICAL NOTE",intro,f1,f2,note,close])),{"section_header_level_1":1,"text":3,"formula":2}

builders={"notice":("Notice",notice),"report":("Report with tables",report),
          "invoice":("Invoice-style document",invoice),"technical_note":("Formula / technical note",technical)}
pages={}
for pid,(kind,builder) in builders.items():
    image,reference,expected=builder()
    path=PAGE_DIR/f"{pid}.png"; image.save(path)
    pages[pid]={"id":pid,"kind":kind,"image":image,"reference_text":reference,
                "expected_counts":expected,"path":str(path)}
print({k:v["image"].size for k,v in pages.items()})

## 9. Native structure and supplemental text helpers

SmolDocling DocTags are parsed only for the element/coordinate/OTSL grammar used by the live carrier. GOT formatted output remains native text and is not coerced into DocTags.

In [ ]:
ELEMENT_TAGS=("section_header_level_1","section_header_level_2","section_header_level_3",
"text","paragraph","list_item","ordered_list","unordered_list","otsl","picture","caption",
"formula","code","page_header","page_footer","footnote","chart","key_value_region")
TAG_RE=re.compile(r"</?([a-z_]+(?:_[0-9]+)?)>")
LOC_RE=re.compile(r"<loc_([0-9]+)>")
OTSL_RE=re.compile(r"<(?:fcel|ecel|ched|rhed|srow|lcel|ucel|xcel|nl)>")
OTSL_TYPES=("fcel","ecel","ched","rhed","srow","lcel","ucel","xcel","nl")

def doctags_to_text(x):
    x=LOC_RE.sub(" ",x); x=OTSL_RE.sub(" ",x); x=TAG_RE.sub(" ",x)
    return normalise_text(x)

def doctags_summary(x):
    opened=Counter(m.group(1) for m in TAG_RE.finditer(x) if not m.group(0).startswith("</"))
    locs=[int(v) for v in LOC_RE.findall(x)]
    if any(v<0 or v>500 for v in locs): raise RuntimeError("loc token outside 0..500")
    return {"counts":{t:opened.get(t,0) for t in ELEMENT_TAGS},
            "n_elements":sum(opened.get(t,0) for t in ELEMENT_TAGS),
            "n_loc_tokens":len(locs),"n_location_groups":len(locs)//4,
            "loc_tokens_divisible_by_4":len(locs)%4==0,
            "n_table_cells":len(OTSL_RE.findall(x)),
            "otsl_token_counts":{t:len(re.findall(fr"<{t}>",x)) for t in OTSL_TYPES},
            "wrapped_in_doctag":x.lstrip().startswith("<doctag>") and x.rstrip().endswith("</doctag>"),
            "n_chars":len(x)}

def token_overlap(reference,hypothesis):
    a,b=Counter(words(reference)),Counter(words(hypothesis))
    overlap=sum((a&b).values())
    p=overlap/sum(b.values()) if b else 0.0
    r=overlap/sum(a.values()) if a else 0.0
    f=2*p*r/(p+r) if p+r else 0.0
    return p,r,f

def page_row(model,page,item,text):
    m=one_metrics(page["reference_text"],text); p,r,f=token_overlap(page["reference_text"],text)
    return {"page_id":page["id"],"page_kind":page["kind"],"model":model,
            "cer":m["cer"],"wer":m["wer"],"token_precision":p,"token_recall":r,"token_f1":f,
            "reference_chars":m["reference_chars"],"hypothesis_chars":m["hypothesis_chars"],
            "length_ratio":m["length_ratio"],"new_tokens":item["new_tokens"],
            "truncated":item["truncated"],"inference_seconds":item.get("inference_seconds")}

def error_category(cer,ratio,empty,truncated):
    if truncated:return "truncated"
    if empty:return "empty"
    if ratio>1.5:return "runaway_or_hallucinated"
    if cer==0:return "exact"
    if cer<0.2:return "mostly_correct"
    if cer<0.8:return "partial"
    return "severe"

## 10. GOT-OCR frozen inference

The wrapper reproduces the live carrier: float32, verified local snapshot, one-image chunks through the expensive vision tower, greedy generation, `<|im_end|>` stop string, and `plain` / `format` modes.

In [ ]:
GOT_STOP="<|im_end|>"
GOT_MAX_NEW_TOKENS=4096

def load_got():
    t=time.perf_counter()
    p=AutoProcessor.from_pretrained(str(GOT_DIR),local_files_only=True,trust_remote_code=False)
    m=AutoModelForImageTextToText.from_pretrained(
        str(GOT_DIR),local_files_only=True,trust_remote_code=False,dtype=DTYPE).eval().to(DEVICE)
    for x in m.parameters(): x.requires_grad_(False)
    pad=p.tokenizer.pad_token_id; stop=p.tokenizer.convert_tokens_to_ids(GOT_STOP)
    original=m.model.get_image_features
    def chunked(pixel_values,**kwargs):
        if pixel_values.shape[0]<=1:return original(pixel_values=pixel_values,**kwargs)
        return torch.cat([original(pixel_values=pixel_values[i:i+1],**kwargs)
                          for i in range(pixel_values.shape[0])],dim=0)
    m.model.get_image_features=chunked
    return m,p,pad,stop,time.perf_counter()-t

def got_generate(model,processor,pad_id,stop_id,images,mode,budget):
    if mode not in ("plain","format"): raise ValueError(mode)
    if not 1<=budget<=GOT_MAX_NEW_TOKENS: raise ValueError(budget)
    inputs=processor(list(images),return_tensors="pt",padding=True,format=(mode=="format"))
    if not bool(inputs["attention_mask"].all()): raise RuntimeError("GOT batch prompts differ in length")
    inputs=inputs.to(DEVICE)
    if torch.cuda.is_available():torch.cuda.synchronize()
    t=time.perf_counter()
    with torch.inference_mode():
        generated=model.generate(**inputs,do_sample=False,tokenizer=processor.tokenizer,
                                 stop_strings=GOT_STOP,max_new_tokens=budget)
    if torch.cuda.is_available():torch.cuda.synchronize()
    elapsed=time.perf_counter()-t; prompt_len=int(inputs["input_ids"].shape[1]); out=[]
    for row in generated:
        ids=row[prompt_len:].tolist(); n=len(ids)
        for pos,token in enumerate(ids):
            if token in (stop_id,pad_id):
                n=pos+(token==stop_id); break
        text=processor.decode(row[prompt_len:prompt_len+n],skip_special_tokens=True)
        out.append({"text":str(text).replace(GOT_STOP,"").strip(),"mode":mode,"new_tokens":int(n),
                    "truncated":n>=budget,"inference_seconds":elapsed/len(images)})
    return out

got_model,got_processor,got_pad,got_stop,got_load_seconds=load_got()
got_parameter_count=sum(p.numel() for p in got_model.parameters())
print({"load_seconds":got_load_seconds,"parameters":got_parameter_count})

## 11. GOT-OCR on held-out Belfort lines

**Predict before running:** will frozen GOT beat the empty-transcript baseline on these handwritten lines? Write a reason based on the domain. Notice CER and hypothesis length together: CER above 1 can reflect insertions, not a programming error.


In [ ]:
got_belfort_items=[]; got_belfort_metrics=None; got_belfort_seconds=0.0
if RUN_BELFORT:
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
    t=time.perf_counter()
    for start in range(0,len(test_records),BATCH_SIZE):
        batch=test_records[start:start+BATCH_SIZE]
        got_belfort_items.extend(got_generate(got_model,got_processor,got_pad,got_stop,
                                               [r["image"] for r in batch],"plain",GOT_LINE_MAX_NEW_TOKENS))
        if (start+BATCH_SIZE)%40==0: print("GOT",min(start+BATCH_SIZE,len(test_records)),"/",len(test_records))
    got_belfort_seconds=time.perf_counter()-t
    got_belfort_metrics=corpus_metrics([x["text"] for x in got_belfort_items],test_records)
    got_belfort_metrics["truncated"]=sum(x["truncated"] for x in got_belfort_items)
    got_belfort_metrics["empty_hypotheses"]=sum(not normalise_text(x["text"]) for x in got_belfort_items)
    got_peak_gpu=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
    print({k:got_belfort_metrics[k] for k in ("cer","wer","cer_macro","exact_match","length_ratio","truncated")})
else:
    got_peak_gpu=None

## 12. GOT-OCR rendered-page capabilities

`plain` is scored against authored visible text. `format` is preserved as native output and is **not** interpreted as DocTags.

In [ ]:
got_page_plain={}; got_page_rows=[]; got_format_outputs={}
if RUN_PAGE_SUITE:
    for pid,page in pages.items():
        item=got_generate(got_model,got_processor,got_pad,got_stop,[page["image"]],"plain",GOT_PAGE_MAX_NEW_TOKENS)[0]
        got_page_plain[pid]=item; got_page_rows.append(page_row("GOT-OCR 2.0",page,item,item["text"]))
    for pid in ("report","invoice","technical_note"):
        item=got_generate(got_model,got_processor,got_pad,got_stop,[pages[pid]["image"]],"format",GOT_PAGE_MAX_NEW_TOKENS)[0]
        text=item["text"]
        got_format_outputs[pid]={**item,"characters":len(text),"lines":len(text.splitlines()),
                                 "format_character_counts":{c:text.count(c) for c in ("|","#","$","\\","_","*")}}
    for r in got_page_rows: print(r["page_id"],round(r["cer"],3),round(r["wer"],3))

## 13. GOT token-budget, shape and hallucination probes

The report is re-run under several output budgets. The notice is distorted to wide/tall shapes and degraded resolution. Blank/noise inputs record generated output without assuming emptiness.

In [ ]:
got_budget_rows=[]; got_shape_rows=[]; got_blank_noise={}
if RUN_TOKEN_BUDGET_EXPERIMENT:
    page=pages["report"]
    for budget in (256,512,1024,2048):
        item=got_generate(got_model,got_processor,got_pad,got_stop,[page["image"]],"plain",budget)[0]
        m=one_metrics(page["reference_text"],item["text"])
        got_budget_rows.append({"model":"GOT-OCR 2.0","page_id":"report","token_budget":budget,
          "generated_tokens":item["new_tokens"],"truncated":item["truncated"],"cer":m["cer"],"wer":m["wer"],
          "text_chars":len(normalise_text(item["text"])),"n_elements":None,"n_table_cells":None})
if RUN_SHAPE_ROBUSTNESS:
    base=pages["notice"]["image"]
    variants={"portrait_original":base,
              "wide":base.resize((1600,700),Image.Resampling.BILINEAR),
              "tall":base.resize((650,1800),Image.Resampling.BILINEAR),
              "low_resolution":base.resize((base.width//2,base.height//2),Image.Resampling.BILINEAR).resize(base.size,Image.Resampling.BILINEAR)}
    for name,image in variants.items():
        item=got_generate(got_model,got_processor,got_pad,got_stop,[image],"plain",GOT_PAGE_MAX_NEW_TOKENS)[0]
        m=one_metrics(pages["notice"]["reference_text"],item["text"])
        got_shape_rows.append({"model":"GOT-OCR 2.0","variant":name,"cer":m["cer"],"wer":m["wer"],
                               "length_ratio":m["length_ratio"],"new_tokens":item["new_tokens"],
                               "truncated":item["truncated"]})
blank=Image.new("RGB",(900,1200),"white")
rng=np.random.default_rng(42)
noise=Image.fromarray(rng.integers(0,256,size=(600,800,3),dtype=np.uint8),"RGB")
for name,image in (("blank",blank),("noise",noise)):
    item=got_generate(got_model,got_processor,got_pad,got_stop,[image],"plain",256)[0]
    got_blank_noise[name]={"text":item["text"],"characters":len(item["text"]),
                           "new_tokens":item["new_tokens"],"truncated":item["truncated"]}
print({k:{x:v[x] for x in ("characters","new_tokens","truncated")} for k,v in got_blank_noise.items()})

## 14. Release GOT before SmolDocling

In [ ]:
del got_model,got_processor
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
print("GOT released.")

## 15. SmolDocling frozen inference

The model uses only the seven supported upstream instruction strings. DocTags are decoded with special markup retained, terminators stripped, and plain text derived by the same carrier logic.

In [ ]:
SMOL_INSTRUCTIONS=("Convert this page to docling.","Convert chart to table.","Convert formula to LaTeX.",
"Convert code to text.","Convert table to OTSL.","Find all 'text' elements on the page, retrieve all section headers.",
"Detect footer elements on the page.")
SMOL_DEFAULT=SMOL_INSTRUCTIONS[0]
SMOL_MAX_NEW_TOKENS=8192
SMOL_TERMINATORS=("<end_of_utterance>","<|im_end|>")

def messages(instruction):
    return [{"role":"user","content":[{"type":"image"},{"type":"text","text":instruction}]}]

def load_smoldoc():
    t=time.perf_counter()
    p=AutoProcessor.from_pretrained(str(SMOL_DIR),local_files_only=True,trust_remote_code=False)
    m=AutoModelForImageTextToText.from_pretrained(
        str(SMOL_DIR),local_files_only=True,trust_remote_code=False,dtype=DTYPE).eval().to(DEVICE)
    for x in m.parameters():x.requires_grad_(False)
    return m,p,p.tokenizer.convert_tokens_to_ids("<end_of_utterance>"),p.tokenizer.pad_token_id,time.perf_counter()-t

def smol_generate(model,processor,end_id,pad_id,images,instruction,budget):
    if instruction not in SMOL_INSTRUCTIONS:raise ValueError(instruction)
    if not 1<=budget<=SMOL_MAX_NEW_TOKENS:raise ValueError(budget)
    prompt=processor.apply_chat_template(messages(instruction),add_generation_prompt=True)
    processor.tokenizer.padding_side="left"
    inputs=processor(text=[prompt]*len(images),images=[[im.convert("RGB")] for im in images],
                     return_tensors="pt",padding=True).to(DEVICE)
    if torch.cuda.is_available():torch.cuda.synchronize()
    t=time.perf_counter()
    with torch.inference_mode(): generated=model.generate(**inputs,max_new_tokens=budget,do_sample=False)
    if torch.cuda.is_available():torch.cuda.synchronize()
    elapsed=time.perf_counter()-t; prompt_len=int(inputs["input_ids"].shape[1]); out=[]
    for row in generated[:,prompt_len:]:
        ids=row.tolist(); n=len(ids)
        for pos,token in enumerate(ids):
            if token in (end_id,pad_id):
                n=pos+(token==end_id);break
        raw=processor.tokenizer.decode(ids[:n],skip_special_tokens=False)
        for term in SMOL_TERMINATORS:raw=raw.replace(term,"")
        raw=raw.strip()
        out.append({"doctags":raw,"text":doctags_to_text(raw),"instruction":instruction,
                    "new_tokens":int(n),"truncated":n>=budget,"inference_seconds":elapsed/len(images)})
    return out

smol_model,smol_processor,smol_end,smol_pad,smol_load_seconds=load_smoldoc()
smol_parameter_count=sum(p.numel() for p in smol_model.parameters())
print({"load_seconds":smol_load_seconds,"parameters":smol_parameter_count})

## 16. SmolDocling on held-out Belfort lines

**Predict before running:** will a model designed for document structure transcribe these cropped handwritten lines better? Predict CER relative to GOT and the empty baseline, then compare the same held-out lines. Architecture or output format alone does not determine the answer.


In [ ]:
smol_belfort_items=[]; smol_belfort_metrics=None; smol_belfort_seconds=0.0
if RUN_BELFORT:
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
    t=time.perf_counter()
    for start in range(0,len(test_records),BATCH_SIZE):
        batch=test_records[start:start+BATCH_SIZE]
        smol_belfort_items.extend(smol_generate(smol_model,smol_processor,smol_end,smol_pad,
          [r["image"] for r in batch],SMOL_DEFAULT,SMOLDOC_LINE_MAX_NEW_TOKENS))
        if (start+BATCH_SIZE)%40==0:print("SmolDocling",min(start+BATCH_SIZE,len(test_records)),"/",len(test_records))
    smol_belfort_seconds=time.perf_counter()-t
    smol_belfort_metrics=corpus_metrics([x["text"] for x in smol_belfort_items],test_records)
    smol_belfort_metrics["truncated"]=sum(x["truncated"] for x in smol_belfort_items)
    smol_belfort_metrics["empty_hypotheses"]=sum(not normalise_text(x["text"]) for x in smol_belfort_items)
    smol_peak_gpu=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
    print({k:smol_belfort_metrics[k] for k in ("cer","wer","cer_macro","exact_match","length_ratio","truncated")})
else:
    smol_peak_gpu=None

## 17. SmolDocling rendered-page text and structure

The default full-page instruction is scored against the same authored visible text as GOT plain mode. DocTags are then analyzed separately for element counts, locations and OTSL table tokens.

In [ ]:
smol_page_results={}; smol_page_rows=[]; smol_structure_rows=[]; smol_summaries={}
if RUN_PAGE_SUITE:
    for pid,page in pages.items():
        item=smol_generate(smol_model,smol_processor,smol_end,smol_pad,[page["image"]],SMOL_DEFAULT,SMOLDOC_PAGE_MAX_NEW_TOKENS)[0]
        smol_page_results[pid]=item; smol_page_rows.append(page_row("SmolDocling",page,item,item["text"]))
        summary=doctags_summary(item["doctags"]); smol_summaries[pid]=summary
        for tag in sorted(set(page["expected_counts"])|set(summary["counts"])):
            expected=int(page["expected_counts"].get(tag,0)); observed=int(summary["counts"].get(tag,0))
            smol_structure_rows.append({"page_id":pid,"element_type":tag,"expected_count":expected,
                                        "observed_count":observed,"absolute_count_error":abs(expected-observed)})
        print(pid,{"CER":round(smol_page_rows[-1]["cer"],3),"WER":round(smol_page_rows[-1]["wer"],3),
                   "elements":summary["n_elements"],"OTSL_cells":summary["n_table_cells"]})

## 18. Native SmolDocling instructions

The report/table, technical-note formula, report headers and notice footer are re-run using the exact supported instruction strings. These are capability demonstrations, not common benchmark outputs.

In [ ]:
specialized_outputs={}
if RUN_PAGE_SUITE:
    demos=[("report","Convert table to OTSL."),
           ("technical_note","Convert formula to LaTeX."),
           ("report","Find all 'text' elements on the page, retrieve all section headers."),
           ("notice","Detect footer elements on the page.")]
    for pid,instruction in demos:
        item=smol_generate(smol_model,smol_processor,smol_end,smol_pad,
                           [pages[pid]["image"]],instruction,SMOLDOC_PAGE_MAX_NEW_TOKENS)[0]
        specialized_outputs[f"{pid}::{instruction}"]=item
        print(pid,instruction,item["new_tokens"],item["truncated"])

## 19. SmolDocling token-budget, shape and hallucination probes

In [ ]:
smol_budget_rows=[]; smol_shape_rows=[]; smol_blank_noise={}
if RUN_TOKEN_BUDGET_EXPERIMENT:
    page=pages["report"]
    for budget in (256,512,1024,2048,4096):
        item=smol_generate(smol_model,smol_processor,smol_end,smol_pad,[page["image"]],SMOL_DEFAULT,budget)[0]
        m=one_metrics(page["reference_text"],item["text"]); s=doctags_summary(item["doctags"])
        smol_budget_rows.append({"model":"SmolDocling","page_id":"report","token_budget":budget,
          "generated_tokens":item["new_tokens"],"truncated":item["truncated"],"cer":m["cer"],"wer":m["wer"],
          "text_chars":len(normalise_text(item["text"])),"n_elements":s["n_elements"],"n_table_cells":s["n_table_cells"]})
if RUN_SHAPE_ROBUSTNESS:
    base=pages["notice"]["image"]
    variants={"portrait_original":base,
              "wide":base.resize((1600,700),Image.Resampling.BILINEAR),
              "tall":base.resize((650,1800),Image.Resampling.BILINEAR),
              "low_resolution":base.resize((base.width//2,base.height//2),Image.Resampling.BILINEAR).resize(base.size,Image.Resampling.BILINEAR)}
    for name,image in variants.items():
        item=smol_generate(smol_model,smol_processor,smol_end,smol_pad,[image],SMOL_DEFAULT,SMOLDOC_PAGE_MAX_NEW_TOKENS)[0]
        m=one_metrics(pages["notice"]["reference_text"],item["text"])
        smol_shape_rows.append({"model":"SmolDocling","variant":name,"cer":m["cer"],"wer":m["wer"],
                                "length_ratio":m["length_ratio"],"new_tokens":item["new_tokens"],
                                "truncated":item["truncated"]})
for name,image in (("blank",blank),("noise",noise)):
    item=smol_generate(smol_model,smol_processor,smol_end,smol_pad,[image],SMOL_DEFAULT,256)[0]
    smol_blank_noise[name]={"text":item["text"],"doctags":item["doctags"],
                            "characters":len(item["text"]),"new_tokens":item["new_tokens"],
                            "truncated":item["truncated"],"structure":doctags_summary(item["doctags"])}
print({k:{x:v[x] for x in ("characters","new_tokens","truncated")} for k,v in smol_blank_noise.items()})

## 20. Cross-model comparison

A diagnostic `good` line is defined as `CER < 0.20`. This threshold is used only to group error cases; it is not a release criterion.

> **What to notice.** Compare the learned-model result with the baseline/reference first, then use the secondary diagnostics to explain the behavior. Do not infer a universal model ranking from one tutorial sample and configuration.

**Checkpoint:** can lower transcription WER prove better table structure?

<details><summary>Sample interpretation</summary>
No. WER compares word sequences, while table elements, order and cell boundaries are different outputs. A plausible table can contain incorrect text. Report both text errors and structural diagnostics on matched inputs, without claiming universal superiority.
</details>


In [ ]:
def summary_row(system,m,truncated=0,empty=0):
    return {"system":system,"cer":m["cer"],"cer_macro":m["cer_macro"],"wer":m["wer"],"wer_macro":m["wer_macro"],
            "exact_match":m["exact_match"],"median_cer":m["median_cer"],"p90_cer":m["p90_cer"],
            "reference_chars":m["ref_chars"],"hypothesis_chars":m["hyp_chars"],
            "length_ratio":m["length_ratio"],"empty_hypotheses":empty,"truncated":truncated}

belfort_result_rows=[]; belfort_metric_rows=[]
if RUN_BELFORT:
    belfort_metric_rows=[summary_row("Empty baseline",empty_metrics),summary_row("Constant baseline",constant_metrics),
      summary_row("GOT-OCR 2.0",got_belfort_metrics,got_belfort_metrics["truncated"],got_belfort_metrics["empty_hypotheses"]),
      summary_row("SmolDocling",smol_belfort_metrics,smol_belfort_metrics["truncated"],smol_belfort_metrics["empty_hypotheses"])]
    groups=Counter()
    for i,r in enumerate(test_records):
        g,s=got_belfort_items[i],smol_belfort_items[i]
        gm,sm=one_metrics(r["text"],g["text"]),one_metrics(r["text"],s["text"])
        if gm["cer"]<.2 and sm["cer"]<.2: cat="both_good"
        elif gm["cer"]<.2: cat="got_only_good"
        elif sm["cer"]<.2: cat="smoldocling_only_good"
        else: cat="both_poor"
        groups[cat]+=1
        for model,item,m in (("GOT-OCR 2.0",g,gm),("SmolDocling",s,sm)):
            belfort_result_rows.append({"sample_id":r["id"],"reference":r["text"],"model":model,
             "hypothesis":item["text"],"cer":m["cer"],"wer":m["wer"],"exact_match":m["exact_match"],
             "reference_chars":m["reference_chars"],"hypothesis_chars":m["hypothesis_chars"],
             "length_ratio":m["length_ratio"],"new_tokens":item["new_tokens"],"truncated":item["truncated"],
             "inference_seconds":item["inference_seconds"],
             "error_category":error_category(m["cer"],m["length_ratio"],not normalise_text(item["text"]),item["truncated"]),
             "disagreement_category":cat})
    print(dict(groups))
    for row in belfort_metric_rows: print(row["system"],round(row["cer"],3),round(row["wer"],3))
page_text_rows=got_page_rows+smol_page_rows

## 21. Try it yourself

On the report page, compare text fidelity with structure. Lower WER does not necessarily imply better table/layout recovery, and a structurally plausible table can still contain wrong cell text.

In [ ]:
for row in page_text_rows:
    print(row["page_id"],row["model"],
          "CER",round(row["cer"],3),"WER",round(row["wer"],3),"token_F1",round(row["token_f1"],3))

## 22. Resource comparison

Model size is only one cost component: visual-token strategy and generated output length also affect runtime.

In [ ]:
resource_rows=[
 {"model":"GOT-OCR 2.0","parameters":got_parameter_count,
  "weight_bytes":next(x["bytes"] for x in GOT_MANIFEST["files"] if x["path"]=="model.safetensors"),
  "input_strategy":"1024x1024 square; aspect ratio not preserved",
  "snapshot_verify_seconds":got_verify_seconds,"load_seconds":got_load_seconds,
  "belfort_seconds":got_belfort_seconds,
  "mean_line_latency_s":got_belfort_seconds/len(test_records) if RUN_BELFORT else None,
  "mean_page_latency_s":float(np.mean([x["inference_seconds"] for x in got_page_plain.values()])) if got_page_plain else None,
  "peak_gpu_memory_bytes":got_peak_gpu},
 {"model":"SmolDocling","parameters":smol_parameter_count,
  "weight_bytes":next(x["bytes"] for x in SMOL_MANIFEST["files"] if x["path"]=="model.safetensors"),
  "input_strategy":"longest edge 2048; 512px tiles + global view",
  "snapshot_verify_seconds":smol_verify_seconds,"load_seconds":smol_load_seconds,
  "belfort_seconds":smol_belfort_seconds,
  "mean_line_latency_s":smol_belfort_seconds/len(test_records) if RUN_BELFORT else None,
  "mean_page_latency_s":float(np.mean([x["inference_seconds"] for x in smol_page_results.values()])) if smol_page_results else None,
  "peak_gpu_memory_bytes":smol_peak_gpu}]
for r in resource_rows:print(r)

## 23. Release SmolDocling before export / BYOD

In [ ]:
del smol_model,smol_processor
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
print("SmolDocling released.")

## 24. Machine-readable exports

The output set preserves the common text benchmark, model-native structure, token-budget/shape probes, blank/noise behavior, page sidecars, resource measurements and provenance.

In [ ]:
OUT_ROOT=Path(OUTPUT_DIR); OUT_ROOT.mkdir(parents=True,exist_ok=True)

def write_csv(path,rows,fields):
    with open(path,"w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader()
        for r in rows:w.writerow({k:r.get(k) for k in fields})

write_csv(OUT_ROOT/"belfort_results.csv",belfort_result_rows,
 ["sample_id","reference","model","hypothesis","cer","wer","exact_match","reference_chars",
  "hypothesis_chars","length_ratio","new_tokens","truncated","inference_seconds","error_category","disagreement_category"])
write_csv(OUT_ROOT/"belfort_metrics.csv",belfort_metric_rows,
 ["system","cer","cer_macro","wer","wer_macro","exact_match","median_cer","p90_cer",
  "reference_chars","hypothesis_chars","length_ratio","empty_hypotheses","truncated"])
write_csv(OUT_ROOT/"page_text_metrics.csv",page_text_rows,
 ["page_id","page_kind","model","cer","wer","token_precision","token_recall","token_f1",
  "reference_chars","hypothesis_chars","length_ratio","new_tokens","truncated","inference_seconds"])
write_csv(OUT_ROOT/"smoldocling_structure.csv",smol_structure_rows,
 ["page_id","element_type","expected_count","observed_count","absolute_count_error"])
write_csv(OUT_ROOT/"token_budget.csv",got_budget_rows+smol_budget_rows,
 ["model","page_id","token_budget","generated_tokens","truncated","cer","wer","text_chars","n_elements","n_table_cells"])
write_csv(OUT_ROOT/"shape_robustness.csv",got_shape_rows+smol_shape_rows,
 ["model","variant","cer","wer","length_ratio","new_tokens","truncated"])
write_csv(OUT_ROOT/"resource_metrics.csv",resource_rows,
 ["model","parameters","weight_bytes","input_strategy","snapshot_verify_seconds","load_seconds",
  "belfort_seconds","mean_line_latency_s","mean_page_latency_s","peak_gpu_memory_bytes"])

(OUT_ROOT/"got_format_outputs.json").write_text(json.dumps(got_format_outputs,indent=2,ensure_ascii=False),encoding="utf-8")
(OUT_ROOT/"smoldocling_summary.json").write_text(json.dumps(smol_summaries,indent=2,ensure_ascii=False),encoding="utf-8")
(OUT_ROOT/"specialized_instruction_outputs.json").write_text(json.dumps(specialized_outputs,indent=2,ensure_ascii=False),encoding="utf-8")
(OUT_ROOT/"blank_noise_probes.json").write_text(json.dumps({"got":got_blank_noise,"smoldocling":smol_blank_noise},
                                                         indent=2,ensure_ascii=False),encoding="utf-8")

# One folder per synthetic page with raw native outputs.
for pid,page in pages.items():
    pdir=PAGE_DIR/pid; pdir.mkdir(exist_ok=True)
    page["image"].save(pdir/"source.png")
    (pdir/"reference.txt").write_text(page["reference_text"],encoding="utf-8")
    if pid in got_page_plain:(pdir/"got_plain.txt").write_text(got_page_plain[pid]["text"],encoding="utf-8")
    if pid in got_format_outputs:(pdir/"got_format.txt").write_text(got_format_outputs[pid]["text"],encoding="utf-8")
    if pid in smol_page_results:
        (pdir/"smoldocling.txt").write_text(smol_page_results[pid]["text"],encoding="utf-8")
        (pdir/"smoldocling.doctags").write_text(smol_page_results[pid]["doctags"],encoding="utf-8")

def compact(m):
    if not isinstance(m,dict):return m
    return {k:v for k,v in m.items() if k!="rows"}

metrics_export={
 "baselines":{"empty":compact(empty_metrics),"constant":compact(constant_metrics)},
 "belfort":{"got":compact(got_belfort_metrics),"smoldocling":compact(smol_belfort_metrics)},
 "rendered_pages":page_text_rows,"smoldocling_structure":smol_summaries,
 "token_budget":got_budget_rows+smol_budget_rows,"shape_robustness":got_shape_rows+smol_shape_rows,
 "resources":resource_rows}
(OUT_ROOT/"metrics.json").write_text(json.dumps(metrics_export,indent=2,ensure_ascii=False),encoding="utf-8")

provenance={
 "notebook_spec":"2.1","profile":"TASK-INFERENCE","pedagogical_mode":"WORKSHOP","standalone":True,
 "got":{"model_id":GOT_MANIFEST["modelId"],"revision":GOT_MANIFEST["revision"],"license":"Apache-2.0",
        "manifest":GOT_MANIFEST,"parameter_count":got_parameter_count,
        "preprocessing":"1024x1024 square; aspect ratio not preserved","decoding":"greedy"},
 "smoldocling":{"model_id":SMOL_MANIFEST["modelId"],"revision":SMOL_MANIFEST["revision"],
        "license":"CDLA-Permissive-2.0","manifest":SMOL_MANIFEST,"parameter_count":smol_parameter_count,
        "preprocessing":"longest edge 2048; 512px tiles + global view","decoding":"greedy"},
 "belfort":{"repo":CORPUS_REPO,"revision":CORPUS_REVISION,"file":CORPUS_FILE,
        "row_group_pins":ROW_GROUP_PINS,"sample_digest":SAMPLE_DIGEST,
        "split_seed":SAMPLE_SEED,"split_sizes":SAMPLE_SPLIT},
 "rendered_pages":{"generator":"Pillow deterministic renderer v1",
        "pages":{k:{"kind":v["kind"],"expected_counts":v["expected_counts"]} for k,v in pages.items()}},
 "runtime":RUNTIME}
(OUT_ROOT/"provenance.json").write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding="utf-8")
print("Exports written.")

## 25. Stable Belfort example panels

Six predetermined held-out indices are exported. They are not selected by model success or failure.

In [ ]:
PANEL_DIR=OUT_ROOT/"belfort_examples"; PANEL_DIR.mkdir(exist_ok=True)
def panel(record,g,s,n):
    image=record["image"].copy().convert("RGB")
    scale=min(1.0,1100/image.width)
    image=image.resize((max(1,int(image.width*scale)),max(1,int(image.height*scale))),Image.Resampling.BILINEAR)
    canvas=Image.new("RGB",(1100,410),"white"); canvas.paste(image,(0,8)); d=ImageDraw.Draw(canvas)
    y=max(150,image.height+20); ff=fnt(17)
    for label,text in (("REF",record["text"]),("GOT",g["text"]),("SMOL",s["text"])):
        d.text((10,y),f"{label}: {text[:150]}",fill="black",font=ff); y+=70
    p=PANEL_DIR/f"example_{n:02d}.png"; canvas.save(p); return str(p)
if RUN_BELFORT:
    indices=[0,23,46,69,92,115]
    print([panel(test_records[i],got_belfort_items[i],smol_belfort_items[i],n)
           for n,i in enumerate(indices,1)])

## 26. BYOD — labelled or unlabelled

Supply a directory or ZIP of page/region images. An optional `transcripts.csv` with `file,text` (and optional `id`) makes CER/WER measurable.

Without transcripts, native outputs are still exported and the evaluation verdict is `not-measurable`.

**Privacy:** do not upload confidential, regulated, personal or otherwise unauthorized documents to a hosted runtime.

Use 1–500 direct page images, sides 16–16384 pixels and at most 4096×4096 total pixels per image. ZIP expansion is limited to 1 GB. ZIPs may contain images and one `transcripts.csv` only; duplicate basenames are rejected and each archive is staged separately. A directory must contain those files directly, without nested folders or unrelated files. `structure.json` is not consumed or evaluated by this branch.

If supplied, transcripts must match filenames exactly with no duplicate, missing or extra rows. IDs must be unique and cannot contain path separators, colons or NUL. Explicit empty text is valid blank-page ground truth: exact match and edit counts remain measurable, but CER/WER and length ratio are undefined (JSON null / empty CSV field) because their reference denominators are zero. Without transcripts, only predictions are reported.

Each successful attempt writes a new `byod/run-*` output directory. Safe ordinal filenames map back to input IDs via `results.csv`; `report.json` records input digests, immutable model manifests, token budgets, runtime and native-output inventory. Processing stays in this runtime. Both models run sequentially and are released before the next model; no DIMER service is called.


In [ ]:
import tempfile
import traceback
BYOD_EXTENSIONS={".png",".jpg",".jpeg",".webp",".tif",".tiff",".bmp"}

def safe_identifier(value):
    value=str(value).strip()
    if not value or value in (".","..") or any(ch in value for ch in ("/", "\\", ":", "\x00")):
        raise ValueError(f"Unsafe or empty BYOD id: {value!r}")
    return value

def load_byod(path):
    src=Path(path)
    source_zip_sha256=None
    if src.is_file() and src.suffix.lower()==".zip":
        source_zip_sha256=sha256_file(src)
        with zipfile.ZipFile(src) as z:
            total=0; members=[]; seen=set()
            for info in z.infolist():
                name=info.filename.replace("\\","/"); parts=[p for p in name.split("/") if p]
                if info.is_dir(): continue
                if name.startswith("/") or ".." in parts or any(":" in part for part in parts):
                    raise ValueError(f"unsafe ZIP member {name}")
                mode=(info.external_attr>>16)&0o170000
                if mode==0o120000: raise ValueError(f"symlink refused {name}")
                total+=info.file_size
                if total>1_000_000_000: raise ValueError("ZIP exceeds 1 GB expanded")
                basename=Path(name).name
                if basename.casefold() in seen: raise ValueError(f"Duplicate ZIP basename: {basename}")
                if Path(basename).suffix.lower() not in BYOD_EXTENSIONS and basename!="transcripts.csv":
                    raise ValueError(f"Unsupported BYOD ZIP file: {basename}; supply images and optional transcripts.csv only")
                seen.add(basename.casefold()); members.append((info,basename))
            root=Path(tempfile.mkdtemp(prefix="ocr-byod-"))
            for info,basename in members: (root/basename).write_bytes(z.read(info))
    elif src.is_dir(): root=src
    else: raise ValueError("BYOD_PATH must be directory or ZIP")
    unexpected=[p.name for p in root.iterdir() if p.is_file() and p.suffix.lower() not in BYOD_EXTENSIONS and p.name!="transcripts.csv"]
    if unexpected: raise ValueError(f"Unsupported BYOD files: {unexpected}; supply images and optional transcripts.csv only")
    if any(p.is_dir() for p in root.iterdir()): raise ValueError("BYOD directories must contain page images directly, not nested folders")
    images=sorted(p for p in root.iterdir() if p.is_file() and p.suffix.lower() in BYOD_EXTENSIONS)
    if not 1<=len(images)<=500: raise ValueError("BYOD supports 1..500 images")
    filenames=[p.name for p in images]
    if len({name.casefold() for name in filenames})!=len(filenames):
        raise ValueError("Duplicate BYOD image filenames")
    transcript_file=root/"transcripts.csv"; refs={}; ids={}; transcript_sha256=None
    labelled=transcript_file.is_file()
    if labelled:
        payload=transcript_file.read_bytes(); transcript_sha256=hashlib.sha256(payload).hexdigest()
        reader=csv.DictReader(io.StringIO(payload.decode("utf-8-sig"))); rows=list(reader)
        if not rows or not {"file","text"}.issubset(reader.fieldnames or []):
            raise ValueError("transcripts.csv requires file,text and exactly one row per image")
        for r in rows:
            name=(r.get("file") or "").strip()
            if not name or name in refs: raise ValueError(f"Duplicate or empty transcript filename: {name!r}")
            if r.get("text") is None: raise ValueError(f"{name}: missing text field; use an explicit empty CSV field for a blank page")
            refs[name]=normalise_text(r["text"])
            ids[name]=safe_identifier(r.get("id") or name)
        missing=sorted(set(filenames)-set(refs)); extra=sorted(set(refs)-set(filenames))
        if missing or extra: raise ValueError(f"Transcript files must match images exactly; missing={missing}, extra={extra}")
    effective_ids=[ids.get(name,name) for name in filenames]
    if len({value.casefold() for value in effective_ids})!=len(effective_ids):
        raise ValueError("BYOD ids must be unique")
    out=[]
    for p,record_id in zip(images,effective_ids,strict=True):
        payload=p.read_bytes()
        with Image.open(io.BytesIO(payload)) as decoded:
            decoded.load(); image=decoded.convert("RGB")
        if min(image.size)<MIN_IMAGE_SIDE or max(image.size)>MAX_IMAGE_SIDE or image.width*image.height>MAX_IMAGE_PIXELS:
            raise ValueError(f"{p.name}: outside image ceilings")
        out.append({"id":record_id,"file":p.name,"image":image,"text":refs.get(p.name),
                    "source_sha256":hashlib.sha256(payload).hexdigest()})
    return out,labelled,{"source_zip_sha256":source_zip_sha256,"transcripts_sha256":transcript_sha256,
                         "images":{r["file"]:r["source_sha256"] for r in out}}

def byod_text_metrics(reference,hypothesis):
    if normalise_text(reference): return one_metrics(reference,hypothesis)
    hyp=normalise_text(hypothesis)
    return {"cer":None,"wer":None,"length_ratio":None,"exact_match":hyp=="",
            "reference_chars":0,"reference_words":0,"hypothesis_chars":len(hyp),
            "char_edits":len(hyp),"word_edits":len(words(hyp)),
            "rate_reason":"Empty reference: CER/WER and length ratio have zero denominators"}

def infer_byod(records):
    gm=gp=None
    try:
        gm,gp,gpad,gstop,_=load_got(); gplain=[]; gformat=[]
        for start in range(0,len(records),BATCH_SIZE):
            batch=records[start:start+BATCH_SIZE]
            gplain.extend(got_generate(gm,gp,gpad,gstop,[r["image"] for r in batch],"plain",GOT_PAGE_MAX_NEW_TOKENS))
        for r in records:
            gformat.append(got_generate(gm,gp,gpad,gstop,[r["image"]],"format",GOT_PAGE_MAX_NEW_TOKENS)[0])
    except Exception as exc:
        traceback.clear_frames(exc.__traceback__)
        raise
    finally:
        gm=gp=None; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    sm=sp=None
    try:
        sm,sp,se,spd,_=load_smoldoc(); sitems=[]
        for start in range(0,len(records),BATCH_SIZE):
            batch=records[start:start+BATCH_SIZE]
            sitems.extend(smol_generate(sm,sp,se,spd,[r["image"] for r in batch],SMOL_DEFAULT,SMOLDOC_PAGE_MAX_NEW_TOKENS))
    except Exception as exc:
        traceback.clear_frames(exc.__traceback__)
        raise
    finally:
        sm=sp=None; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    for label,items in (("GOT plain",gplain),("GOT format",gformat),("SmolDocling",sitems)):
        if len(items)!=len(records): raise RuntimeError(f"{label}: output count does not match input images")
    return gplain,gformat,sitems

byod_report=None
if USE_BYOD:
    records,labelled,byod_input=load_byod(BYOD_PATH)
    gplain,gformat,sitems=infer_byod(records)
    byod_parent=OUT_ROOT/"byod"; byod_parent.mkdir(parents=True,exist_ok=True)
    byod_dir=Path(tempfile.mkdtemp(prefix="run-",dir=byod_parent))
    rows=[]; output_files=[]
    for i,r in enumerate(records):
        stem=f"page-{i+1:04d}"
        for model,item in (("GOT-OCR 2.0",gplain[i]),("SmolDocling",sitems[i])):
            row={"id":r["id"],"file":r["file"],"output_stem":stem,"model":model,"text":item["text"],
                 "new_tokens":item["new_tokens"],"truncated":item["truncated"]}
            if labelled: row.update(byod_text_metrics(r["text"],item["text"]))
            rows.append(row)
        for suffix,text in (("got_plain.txt",gplain[i]["text"]),("got_format.txt",gformat[i]["text"]),
                            ("smoldocling.txt",sitems[i]["text"]),("smoldocling.doctags",sitems[i]["doctags"])):
            output=byod_dir/f"{stem}.{suffix}"; output.write_text(text,encoding="utf-8"); output_files.append(output.name)
    fields=["id","file","output_stem","model","text","new_tokens","truncated"]
    if labelled: fields += ["cer","wer","exact_match","reference_chars","hypothesis_chars","length_ratio","char_edits","word_edits","reference_words","rate_reason"]
    write_csv(byod_dir/"results.csv",rows,fields)
    byod_report={"images":len(records),"evaluation_verdict":"measured" if labelled else "not-measurable",
                 "undefined_rate_rows":sum(r.get("rate_reason") is not None for r in rows),
                 "output_directory":str(byod_dir),"native_output_files":output_files,
                 "input":byod_input,"runtime":RUNTIME,
                 "models":{"got":GOT_MANIFEST,"smoldocling":SMOL_MANIFEST},
                 "generation":{"got_page_max_new_tokens":GOT_PAGE_MAX_NEW_TOKENS,
                               "smoldocling_page_max_new_tokens":SMOLDOC_PAGE_MAX_NEW_TOKENS,"decoding":"greedy"}}
    (byod_dir/"report.json").write_text(json.dumps(byod_report,indent=2,ensure_ascii=False),encoding="utf-8")
    print({k:byod_report[k] for k in ("images","evaluation_verdict","undefined_rate_rows","output_directory")})
else:
    print("BYOD disabled.")


## 27. Interpretation and limitations

- OCR fidelity and structural fidelity are distinct.
- GOT `format` output is not DocTags.
- Reading-order differences can inflate WER even when many visible tokens are recovered; token-F1 is supplemental.
- Larger token budgets reduce some truncation but can also allow repetition or hallucination.
- Both systems are generative and provide no calibrated correctness score.
- Belfort is French handwriting; the rendered pages are synthetic.
- SmolDocling is the exact pinned `preview` checkpoint, not a later Docling/Granite successor.
- Neither notebook result substitutes for deployment-domain labelled evaluation.

## 28. Terminal summary

In [ ]:
required=[OUT_ROOT/"belfort_results.csv",OUT_ROOT/"belfort_metrics.csv",OUT_ROOT/"page_text_metrics.csv",
 OUT_ROOT/"smoldocling_structure.csv",OUT_ROOT/"token_budget.csv",OUT_ROOT/"shape_robustness.csv",
 OUT_ROOT/"resource_metrics.csv",OUT_ROOT/"got_format_outputs.json",OUT_ROOT/"smoldocling_summary.json",
 OUT_ROOT/"specialized_instruction_outputs.json",OUT_ROOT/"metrics.json",OUT_ROOT/"provenance.json"]
if USE_BYOD:
    required.extend([byod_dir/"results.csv",byod_dir/"report.json"])
    required.extend(byod_dir/name for name in byod_report["native_output_files"])
missing=[str(p) for p in required if not p.is_file()]
if missing:raise RuntimeError(f"missing outputs: {missing}")

print("DIMER OCR & Document Extraction Notebook")
print("-"*44)
print(f"Belfort held-out lines: {len(test_records)}")
if RUN_BELFORT:
    print(f"{'System':<22} {'CER':>8} {'WER':>8} {'Exact':>8}")
    for r in belfort_metric_rows:
        print(f"{r['system']:<22} {r['cer']:>8.3f} {r['wer']:>8.3f} {r['exact_match']:>8.3f}")
print(f"Rendered pages: {len(pages)}")
for model in ("GOT-OCR 2.0","SmolDocling"):
    rows=[r for r in page_text_rows if r["model"]==model]
    if rows:
        print(model, "mean CER",round(float(np.mean([r["cer"] for r in rows])),3),
              "mean WER",round(float(np.mean([r["wer"] for r in rows])),3),
              "truncated",sum(r["truncated"] for r in rows))
if smol_summaries:print("SmolDocling total OTSL cells:",sum(x["n_table_cells"] for x in smol_summaries.values()))
print("Outputs:",OUT_ROOT)

## Try it yourself — one controlled change

Use the existing token-budget probes, which run by default. Do not rerun released-model cells after the canonical run.

1. **Predict:** before viewing Sections 13 and 19, predict whether raising GOT's report-page budget from 256 to 512 will lower CER or only lengthen its output. State why a longer output could also add errors.
2. **Change one thing:** compare the existing 256 and 512 rows for GOT on the same report image. The only experimental factor is the generation budget; model, image, mode and reference stay fixed. Keep the canonical page budget unchanged.
3. **Run:** execute Sections 13 and 19 once in their original order while their respective models are loaded. If probes were disabled, use a separate fresh notebook/runtime with `RUN_TOKEN_BUDGET_EXPERIMENT=True`; preserve the original exports. Do not rerun Configuration or model-loading cells inside a completed run.
4. **Observe:** record budget, generated tokens, truncation flag, CER and WER from the two GOT rows. Separately compare the same two budgets for SmolDocling, including element/cell counts; these do not substitute for text accuracy.
5. **Explain:** "I predicted __. At 256 versus 512 tokens, CER changed from __ to __ and truncation from __ to __. This suggests __ on this page, but __ limits the conclusion."

Use the existing output tables or exported `token_budget.csv`; reading them cannot change canonical arrays, model settings or report files. These pages were already inspected, so this is exploratory evidence. Do not choose a budget here and report the same page as an untouched confirmation set.


## Self-paced checkpoint

Before opening the sample interpretation, answer:

1. What did the model/system receive as input, and what did it produce?
2. Which baseline/reference tells you whether the learned model added value?
3. What failure mode or tradeoff matters most here?
4. What additional evidence would you want before transferring the result to a new domain?

<details>
<summary><b>Show a sample interpretation</b></summary>

OCR text accuracy and document-structure quality are related but not identical. CER/WER can diagnose transcription errors, while a structured extractor can still fail in layout or field organization. Hallucination and token-budget probes are important because fluent output is not evidence that every extracted token is present in the page.

Use the outputs from **your run** when writing your final answer; small numeric differences across supported runtimes are possible.

</details>


## Write an evidence-based conclusion

1. **State the question** tested by this notebook.
2. **Report the primary result** against the relevant baseline/reference.
3. **Add supporting evidence** from a secondary metric, error pattern, disagreement, or qualitative diagnostic.
4. **Account for cost/complexity** when it materially affects the comparison.
5. **State the limits** of the data, split, model revision, and configuration.

Compare GOT-OCR and SmolDocling only on the capabilities evaluated under the common protocol, report the relevant text metric and structural/qualitative evidence, identify one hallucination or page-shape failure mode, and state which outputs require further verification before operational use.


# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or execution is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run top-to-bottom. |
| Package/version or stale-module error | Incompatible libraries were already imported in the hosted kernel | Start a fresh runtime and choose **Run all** before importing extra packages. Do not bypass version checks. |
| Model/sample digest or byte-size check fails | Download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory or runtime restart | Too many large models/intermediates are resident | Use the default tier, follow explicit unload/release steps, and avoid combining optional heavy branches. |
| BYOD validation fails | Input does not satisfy the documented schema, shape, labels, or limits | Follow the validation message, correct the indicated field/format, then rerun the BYOD branch. |
| Your numbers differ slightly | Supported hardware/library execution can introduce small numerical variation | Verify the split, model revision, metric definition, and qualitative pattern before treating the difference as substantive. |


# Glossary

| Term | Meaning in this notebook |
|---|---|
| **OCR** | Optical character recognition: converting visible text in an image into machine-readable text. |
| **CER** | Character Error Rate; edit distance normalized by reference characters. |
| **WER** | Word Error Rate; edit distance normalized by reference words. |
| **Structured extraction** | Recovering organization such as sections, fields, tables, or markup rather than plain text alone. |
| **Token budget** | The maximum amount of text/model output that can be processed or generated in one pass. |
| **Hallucination** | Output content that is not supported by the source document. |